# Init Configuration

In [ ]:
import os
import pandas as pd
from pymongo import MongoClient, UpdateOne, ASCENDING
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display, HTML, Markdown
import math
from typing import Dict, List, Any, Tuple, Optional
from dotenv import load_dotenv
import sys
from datetime import datetime
sys.path.append("/home/ubuntu/projects/AI/git/users/huynq/V03-Sync-Services")
from constants import MongoDBConfig, MigrateConfig
load_dotenv()
client = MongoClient(
    host=MongoDBConfig.HOST,
    port=MongoDBConfig.PORT,
    username=MongoDBConfig.USERNAME,
    password=MongoDBConfig.PASSWORD,
    serverSelectionTimeoutMS=30000,
    connectTimeoutMS=30000
)

SOURCE_CORE_DB = "v03_core_281125_dev"
TARGET_CORE_DB = "v03_core_11032026"
EFFECTIVE_STATUS_COLLECTION = "law_effective_status"
effective_status_collection = client[TARGET_CORE_DB][EFFECTIVE_STATUS_COLLECTION]

In [2]:
def _to_str(value):
    if value is None:
        return ""
    return str(value)

def _to_int(value, default=0):
    if value is None or value == "":
        return default
    try:
        return int(value)
    except Exception:
        return default


def _to_float(value, default=0.0):
    if value is None or value == "":
        return default
    try:
        return float(value)
    except Exception:
        return default


def _format_datetime(value):
    if value is None or value == "":
        return ""
    if isinstance(value, datetime):
        return value.strftime("%Y-%m-%d %H:%M:%S")
    if isinstance(value, (int, float)):
        try:
            return datetime.fromtimestamp(value).strftime("%Y-%m-%d %H:%M:%S")
        except Exception:
            return ""
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return ""
        try:
            return datetime.fromtimestamp(float(value)).strftime("%Y-%m-%d %H:%M:%S")
        except Exception:
            pass
        for fmt in (
            "%Y-%m-%d %H:%M:%S",
            "%Y-%m-%d",
            "%Y-%m-%dT%H:%M:%S",
            "%Y-%m-%dT%H:%M:%S.%f",
            "%Y-%m-%dT%H:%M:%S%z",
            "%Y-%m-%dT%H:%M:%S.%f%z",
        ):
            try:
                dt = datetime.strptime(value, fmt)
                return dt.strftime("%Y-%m-%d %H:%M:%S")
            except Exception:
                continue
        try:
            return datetime.fromisoformat(value.replace("Z", "+00:00")).strftime("%Y-%m-%d %H:%M:%S")
        except Exception:
            return value
    return ""


def _normalize_steps(steps):
    if not isinstance(steps, list):
        return [""]
    if not steps:
        return [""]
    normalized = []
    for step in steps:
        if not isinstance(step, dict):
            continue
        normalized.append({
            "step": _to_str(step.get("step")),
            "start_time": _format_datetime(step.get("start_time")),
            "end_time": _format_datetime(step.get("end_time")),
            "duration": _to_float(step.get("duration")),
            "status": _to_str(step.get("status")),
            "error_message": _to_str(step.get("error_message"))
        })
    return normalized if normalized else [""]


def _normalize_result(old_document):
    result = old_document.get("result")
    results = old_document.get("results")

    if result not in (None, "", [], {}):
        return result

    if results not in (None, "", [], {}):
        return results

    return ""

def _normalize_string_array(values):
    if not isinstance(values, list):
        return [""]
    cleaned = []
    for value in values:
        value_str = _to_str(value).strip()
        if value_str:
            cleaned.append(value_str)
    return cleaned if cleaned else [""]


def _normalize_status(value):
    if isinstance(value, dict):
        return value
    return {}

def _normalize_config(value):
    if isinstance(value, dict):
        return value
    return {}

def _normalize_datetime(value):
    formatted = _format_datetime(value)
    if not formatted:
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return formatted

def _normalize_metrics(value):
    if not isinstance(value, dict):
        return {}

    normalized = dict(value)
    normalized["trained_at"] = _format_datetime(normalized.get("trained_at"))
    return normalized

def _normalize_bool(value):
    if isinstance(value, bool):
        return value
    return False

def _normalize_json_object(value):
    if isinstance(value, dict):
        return value
    return {}

def _normalize_datetime_reference(value):
    if value is None or value == "":
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if isinstance(value, dict) and "$date" in value:
        try:
            return datetime.fromisoformat(value["$date"].replace("Z", "+00:00")).strftime("%Y-%m-%d %H:%M:%S")
        except Exception:
            return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if isinstance(value, datetime):
        return value.strftime("%Y-%m-%d %H:%M:%S")
    if isinstance(value, (int, float)):
        try:
            return datetime.fromtimestamp(value).strftime("%Y-%m-%d %H:%M:%S")
        except Exception:
            return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        for fmt in (
            "%Y-%m-%d %H:%M:%S",
            "%Y-%m-%d",
            "%H:%M:%S %d/%m/%y",
            "%Y-%m-%dT%H:%M:%S",
            "%Y-%m-%dT%H:%M:%S.%f",
            "%Y-%m-%dT%H:%M:%S%z",
            "%Y-%m-%dT%H:%M:%S.%f%z",
        ):
            try:
                return datetime.strptime(value, fmt).strftime("%Y-%m-%d %H:%M:%S")
            except Exception:
                continue
        try:
            return datetime.fromisoformat(value.replace("Z", "+00:00")).strftime("%Y-%m-%d %H:%M:%S")
        except Exception:
            return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def _load_effective_status_map():
    result = {}
    for doc in effective_status_collection.find({}, {"effective_status_name": 1, "effective_status_id": 1}):
        name = _to_str(doc.get("effective_status_name")).strip()
        eid = _to_str(doc.get("effective_status_id")).strip()
        if name and eid:
            result[name] = eid
    return result

effective_status_map = _load_effective_status_map()
unknown_effective_status_id = effective_status_map.get("Không xác định", "b04750de-31f5-4266-b5c7-ac56c2bac946")

effective_status_map = {
    "Không xác định": "b04750de-31f5-4266-b5c7-ac56c2bac946",
    "Hết hiệu lực": "a2e5eb7f-140b-43e9-9a9e-0b351466ae05",
    "Còn hiệu lực": "3969bc0a-a285-4a6d-9865-5b549cf88d20"
}

def _parse_datetime_flexible(value):
    if not value:
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if isinstance(value, dict) and "$date" in value:
        try:
            dt = datetime.fromisoformat(value["$date"].replace("Z", "+00:00"))
            return dt.strftime("%Y-%m-%d %H:%M:%S")
        except:
            return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if isinstance(value, str):
        try:
            dt = datetime.strptime(value, "%H:%M:%S %d/%m/%y")
            return dt.strftime("%Y-%m-%d %H:%M:%S")
        except:
            try:
                dt = datetime.strptime(value, "%Y-%m-%d %H:%M:%S")
                return dt.strftime("%Y-%m-%d %H:%M:%S")
            except:
                return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [3]:
def parallel_migrate(
    sources,
    mapping_function,
    upsert_key,
    target_collection,
    batch_size=2000,
    max_workers=None
):

    if max_workers is None:
        max_workers = min(16, (os.cpu_count() or 4) * 2)
        
    def process_batch(batch):
        operations = []
        for old_document in batch:
            new_document = mapping_function(old_document)
            key = new_document.get(upsert_key)
            if not key:
                continue
            operations.append(
                UpdateOne(
                    {upsert_key: key},
                    {"$set": new_document},
                    upsert=True
                )
            )

        if operations:
            return target_collection.bulk_write(operations, ordered=False)

        return None


    processed = 0
    errors = []
    total_docs = 0

    for source_collection in sources:
        cursor = source_collection.find({}, no_cursor_timeout=True)
        documents = list(cursor)
        total_docs += len(documents)
        batches = [
            documents[i:i + batch_size]
            for i in range(0, len(documents), batch_size)
        ]

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [
                executor.submit(process_batch, batch)
                for batch in batches
            ]
            for future in as_completed(futures):
                try:
                    result = future.result()
                    if result:
                        processed += result.upserted_count + result.modified_count
                except Exception as e:
                    errors.append(str(e))

    return {
        "source_total": total_docs,
        "processed": processed,
        "errors": len(errors),
        "target_total": target_collection.count_documents({})
    }
    

def parallel_migrate_composite_key(sources, mapping_function, upsert_filter_function, target_collection, batch_size=500, max_workers=4):
    if max_workers is None:
        max_workers = min(16, (os.cpu_count() or 4) * 2)

    def process_batch(batch):
        operations = []
        for old_document in batch:
            new_document = mapping_function(old_document)
            upsert_filter = upsert_filter_function(new_document)
            if not upsert_filter:
                continue
            operations.append(UpdateOne(upsert_filter, {"$set": new_document}, upsert=True))
        if operations:
            return target_collection.bulk_write(operations, ordered=False)
        return None

    processed = 0
    errors = []
    total_docs = 0

    for source_collection in sources:
        cursor = source_collection.find({}, no_cursor_timeout=True)
        documents = list(cursor)
        total_docs += len(documents)
        batches = [documents[i:i + batch_size] for i in range(0, len(documents), batch_size)]

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [executor.submit(process_batch, batch) for batch in batches]
            for future in as_completed(futures):
                try:
                    result = future.result()
                    if result:
                        processed += result.upserted_count + result.modified_count
                except Exception as e:
                    errors.append(str(e))

    return {
        "source_total": total_docs,
        "processed": processed,
        "errors": len(errors),
        "target_total": target_collection.count_documents({})
    }
    
def streaming_parallel_migrate(sources, mapping_function, upsert_key, target_collection, batch_size=5000, max_workers=None):
    if max_workers is None:
        max_workers = min(16, (os.cpu_count() or 4))
    def process_batch(batch):
        operations = []
        skipped_missing_key = 0
        for old_document in batch:
            new_document = mapping_function(old_document)
            key = new_document.get(upsert_key)
            if not key:
                skipped_missing_key += 1
                continue
            operations.append(UpdateOne({upsert_key: key}, {"$set": new_document}, upsert=True))
        if not operations:
            return {"processed": 0, "upserted": 0, "modified": 0, "matched": 0, "skipped_missing_key": skipped_missing_key, "errors": []}
        try:
            result = target_collection.bulk_write(operations, ordered=False)
            return {
                "processed": len(operations),
                "upserted": result.upserted_count,
                "modified": result.modified_count,
                "matched": result.matched_count,
                "skipped_missing_key": skipped_missing_key,
                "errors": []
            }
        except Exception as e:
            return {
                "processed": len(operations),
                "upserted": 0,
                "modified": 0,
                "matched": 0,
                "skipped_missing_key": skipped_missing_key,
                "errors": [str(e)]
            }
    overall = {
        "draft_total": 0,
        "main_total": 0,
        "processed_draft": 0,
        "processed_main": 0,
        "upserted_draft": 0,
        "upserted_main": 0,
        "modified_draft": 0,
        "modified_main": 0,
        "matched_draft": 0,
        "matched_main": 0,
        "skipped_missing_key_draft": 0,
        "skipped_missing_key_main": 0,
        "errors": []
    }
    for source_name, source_collection in sources:
        total_docs = source_collection.count_documents({})
        if source_name == "draft":
            overall["draft_total"] = total_docs
        else:
            overall["main_total"] = total_docs
        futures = []
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            batch = []
            cursor = source_collection.find({}, no_cursor_timeout=True).batch_size(batch_size)
            try:
                for doc in cursor:
                    batch.append(doc)
                    if len(batch) >= batch_size:
                        futures.append(executor.submit(process_batch, batch))
                        batch = []
                if batch:
                    futures.append(executor.submit(process_batch, batch))
            finally:
                cursor.close()
            for future in as_completed(futures):
                result = future.result()
                overall["errors"].extend(result["errors"])
                if source_name == "draft":
                    overall["processed_draft"] += result["processed"]
                    overall["upserted_draft"] += result["upserted"]
                    overall["modified_draft"] += result["modified"]
                    overall["matched_draft"] += result["matched"]
                    overall["skipped_missing_key_draft"] += result["skipped_missing_key"]
                else:
                    overall["processed_main"] += result["processed"]
                    overall["upserted_main"] += result["upserted"]
                    overall["modified_main"] += result["modified"]
                    overall["matched_main"] += result["matched"]
                    overall["skipped_missing_key_main"] += result["skipped_missing_key"]
    return overall

## 1. biz_performance

In [ ]:
TARGET_COLLECTION = "biz_performance"

source_collection = client[SOURCE_CORE_DB][TARGET_COLLECTION]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]


def mapping_biz_performance(old_document):
    base_doc_id = (
        old_document.get("base_doc_id")
        or old_document.get("base_document_id")
        or old_document.get("doc_id")
        or ""
    )

    new_document = {
        "app_request_id": _to_str(old_document.get("app_request_id")),
        "base_doc_id": _to_str(base_doc_id),
        "consumer_id": _to_str(old_document.get("consumer_id")),
        "created_at": _format_datetime(old_document.get("created_at")),
        "duration": _to_float(old_document.get("duration", old_document.get("total_duration", 0.0))),
        "end_time": _format_datetime(old_document.get("end_time")),
        "request_id": _to_str(old_document.get("request_id")),
        "result": _normalize_result(old_document),
        "start_time": _format_datetime(old_document.get("start_time")),
        "status": _to_str(old_document.get("status")),
        "steps": _normalize_steps(old_document.get("steps")),
        "total_duration": _to_float(old_document.get("total_duration")),
        "type_validate": _to_str(old_document.get("type_validate")),
        "validate_doc_id": _to_str(old_document.get("validate_document_id")),
        "version": _to_str(old_document.get("version"))
    }

    return new_document


documents = list(source_collection.find({}))

processed = 0
upserted = 0
errors = []

for old_document in documents:
    try:
        source_id = old_document.get("_id")
        if source_id is None:
            errors.append({
                "source_id": "",
                "request_id": _to_str(old_document.get("request_id")),
                "error": "Missing _id"
            })
            continue

        new_document = mapping_biz_performance(old_document)

        result = target_collection.update_one(
            {"_id": source_id},
            {"$set": new_document},
            upsert=True
        )

        processed += 1
        if result.upserted_id is not None or result.modified_count > 0 or result.matched_count > 0:
            upserted += 1

    except Exception as e:
        errors.append({
            "source_id": _to_str(old_document.get("_id")),
            "request_id": _to_str(old_document.get("request_id")),
            "error": str(e)
        })

print({
    "collection": TARGET_COLLECTION,
    "source_total": len(documents),
    "processed": processed,
    "upserted": upserted,
    "errors": len(errors),
    "target_total": target_collection.count_documents({})
})

if errors:
    display(pd.DataFrame(errors))

{'collection': 'biz_performance', 'source_total': 3410, 'processed': 3410, 'upserted': 3410, 'errors': 0, 'target_total': 3410}


## 2. biz_review_records

In [ ]:
TARGET_COLLECTION = "biz_review_records"

source_collection = client[SOURCE_CORE_DB][TARGET_COLLECTION]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]

def mapping_biz_review_records(old_document):
    new_document = {
        "record_id": _to_str(old_document.get("record_id")),
        "request_id": _to_str(old_document.get("request_id")),
        "app_request_id": _to_str(old_document.get("app_request_id")),
        "user_id": _to_str(old_document.get("user_id")),
        "base_doc_id": _to_str(old_document.get("doc_id")),
        "validate_doc_id": _to_str(old_document.get("validate_document_id")),
        "storage_id": _to_str(old_document.get("storage_code")),
        "review_types": _normalize_string_array(old_document.get("review_types")),
        "status": _normalize_status(old_document.get("status")),
        "created_at": _format_datetime(old_document.get("created_date")),
        "created_by": _to_str(old_document.get("created_by")),
        "last_modified_at": _format_datetime(old_document.get("last_modified")),
        "last_modified_by": _to_str(old_document.get("last_modified_by"))
    }
    return new_document


documents = list(source_collection.find({}))

new_documents = []
errors = []

for old_document in documents:
    try:
        new_document = mapping_biz_review_records(old_document)
        new_documents.append(new_document)
    except Exception as e:
        errors.append({
            "source_id": _to_str(old_document.get("_id")),
            "record_id": _to_str(old_document.get("record_id")),
            "request_id": _to_str(old_document.get("request_id")),
            "error": str(e)
        })

inserted = 0

if new_documents:
    result = target_collection.insert_many(new_documents, ordered=False)
    inserted = len(result.inserted_ids)

print({
    "collection": TARGET_COLLECTION,
    "source_total": len(documents),
    "mapped": len(new_documents),
    "inserted": inserted,
    "errors": len(errors),
    "target_total": target_collection.count_documents({})
})

if errors:
    display(pd.DataFrame(errors))

{'collection': 'biz_review_records', 'source_total': 2453, 'mapped': 2453, 'inserted': 2453, 'errors': 0, 'target_total': 2453}


## 3. law_summary

In [10]:
SOURCE_COLLECTION = "biz_summary"
TARGET_COLLECTION = "law_summary"

source_collection = client[SOURCE_CORE_DB][SOURCE_COLLECTION]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]

# Upsert key dùng bộ (doc_id, summary_type, version) theo schema chuẩn mới.
# Lưu ý: source hiện có 4 bản ghi trùng bộ key này. Tạm coi đó là dữ liệu erroneous.
# Với logic upsert hiện tại, các bản ghi trùng key sẽ bị ghi đè theo thứ tự xử lý cuối cùng.


def mapping_law_summary(old_document):
    new_document = {
        "doc_id": _to_str(old_document.get("doc_id")),
        "summary_type": _to_str(old_document.get("summary_type")),
        "summary_content": _to_str(old_document.get("summary_content")),
        "version": _to_str(old_document.get("version")),
        "created_at": _format_datetime(old_document.get("created_at")),
        "last_modified_at": _format_datetime(old_document.get("last_modified"))
    }
    return new_document


documents = list(source_collection.find({}))

processed = 0
upserted = 0
errors = []

for old_document in documents:
    try:
        new_document = mapping_law_summary(old_document)

        upsert_filter = {
            "doc_id": new_document["doc_id"],
            "summary_type": new_document["summary_type"],
            "version": new_document["version"]
        }

        result = target_collection.update_one(
            upsert_filter,
            {"$set": new_document},
            upsert=True
        )

        processed += 1
        if result.upserted_id is not None or result.modified_count > 0 or result.matched_count > 0:
            upserted += 1

    except Exception as e:
        errors.append({
            "source_id": _to_str(old_document.get("_id")),
            "doc_id": _to_str(old_document.get("doc_id")),
            "summary_type": _to_str(old_document.get("summary_type")),
            "version": _to_str(old_document.get("version")),
            "error": str(e)
        })

print({
    "source_collection": SOURCE_COLLECTION,
    "target_collection": TARGET_COLLECTION,
    "source_total": len(documents),
    "processed": processed,
    "upserted": upserted,
    "errors": len(errors),
    "target_total": target_collection.count_documents({})
})

if errors:
    display(pd.DataFrame(errors))

{'source_collection': 'biz_summary', 'target_collection': 'law_summary', 'source_total': 4307, 'processed': 4307, 'upserted': 4307, 'errors': 0, 'target_total': 4303}


## 4. biz_training_process

In [ ]:
TARGET_COLLECTION = "biz_training_process"

source_collection = client[SOURCE_CORE_DB][TARGET_COLLECTION]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]

# Lưu ý:
# - Upsert theo train_id theo schema chuẩn mới.
# - Field updated_at từ dữ liệu cũ không tiếp tục sử dụng làm last_modified_at trong schema chuẩn hóa.
# - Theo rule migrate hiện tại, toàn bộ last_modified_at sẽ được đưa về "".


def mapping_biz_training_process(old_document):
    new_document = {
        "train_id": _to_str(old_document.get("train_id")),
        "train_name": _to_str(old_document.get("train_name")),
        "tree_id": _to_str(old_document.get("tree_id")),
        "model_id": _to_str(old_document.get("model_id")),
        "model_path": _to_str(old_document.get("model_path")),
        "description": _to_str(old_document.get("description")),
        "status": _to_str(old_document.get("status")),
        "use_status": _to_str(old_document.get("use_status")),
        "dataset_ratio": _to_float(old_document.get("dataset_ratio")),
        "training_duration": _to_float(old_document.get("training_duration")),
        "config": _normalize_config(old_document.get("config")),
        "metrics": _normalize_metrics(old_document.get("metrics")),
        "created_at": _format_datetime(old_document.get("created_at")),
        "created_by": _to_str(old_document.get("created_by")),
        "last_modified_at": "",
        "last_modified_by": _to_str(old_document.get("updated_by"))
    }
    return new_document


documents = list(source_collection.find({}))

processed = 0
upserted = 0
errors = []

for old_document in documents:
    try:
        new_document = mapping_biz_training_process(old_document)
        train_id = new_document["train_id"]

        if not train_id:
            errors.append({
                "source_id": _to_str(old_document.get("_id")),
                "error": "Missing train_id"
            })
            continue

        result = target_collection.update_one(
            {"train_id": train_id},
            {"$set": new_document},
            upsert=True
        )

        processed += 1
        if result.upserted_id is not None or result.modified_count > 0 or result.matched_count > 0:
            upserted += 1

    except Exception as e:
        errors.append({
            "source_id": _to_str(old_document.get("_id")),
            "train_id": _to_str(old_document.get("train_id")),
            "error": str(e)
        })

print({
    "collection": TARGET_COLLECTION,
    "source_total": len(documents),
    "processed": processed,
    "upserted": upserted,
    "errors": len(errors),
    "target_total": target_collection.count_documents({})
})

if errors:
    display(pd.DataFrame(errors))

{'collection': 'biz_training_process', 'source_total': 2, 'processed': 2, 'upserted': 2, 'errors': 0, 'target_total': 2}


## 5. law_documents

In [ ]:
SOURCE_DB = "v03_core_281125_dev"
SOURCE_MAIN = "law_documents"
SOURCE_UPLOAD = "biz_upload_documents"
TARGET_COLLECTION = "law_documents"

source_main = client[SOURCE_DB][SOURCE_MAIN]
source_upload = client[SOURCE_DB][SOURCE_UPLOAD]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]

def _normalize_array(values):
    if not isinstance(values, list):
        return [""]
    cleaned = []
    seen = set()
    for value in values:
        value_str = _to_str(value).strip()
        if value_str and value_str not in seen:
            cleaned.append(value_str)
            seen.add(value_str)
    return cleaned if cleaned else [""]

def _normalize_business_datetime(value):
    return _format_datetime(value)

def _is_valid_law_document_payload(new_document):
    if not _to_str(new_document.get("doc_id")).strip():
        return False
    if not _to_str(new_document.get("doc_code")).strip():
        return False
    if not _to_str(new_document.get("doc_title")).strip():
        return False
    if not _to_str(new_document.get("doc_content")).strip():
        return False
    return True

def mapping_law_documents_main(old_document):
    doc_short_description = _to_str(old_document.get("doc_short_description")).strip()
    if not doc_short_description:
        doc_short_description = _to_str(old_document.get("short_description")).strip()
    if not doc_short_description:
        doc_short_description = _to_str(old_document.get("doc_title"))
    return {
        "doc_id": _to_str(old_document.get("doc_id")),
        "doc_code": _to_str(old_document.get("doc_code")),
        "doc_title": _to_str(old_document.get("doc_title")),
        "doc_short_description": doc_short_description,
        "doc_content": _to_str(old_document.get("doc_content")),
        "doc_issue_date": _normalize_business_datetime(old_document.get("doc_issue_date")),
        "doc_effective_date": _normalize_business_datetime(old_document.get("doc_effective_date")),
        "doc_expiry_date": _normalize_business_datetime(old_document.get("doc_expiry_date")),
        "data_source": _to_str(old_document.get("data_source")),
        "created_at": _normalize_datetime(old_document.get("created_date")),
        "created_by": _to_str(old_document.get("created_by")),
        "last_modified_at": _normalize_datetime(old_document.get("last_modified")),
        "last_modified_by": _to_str(old_document.get("last_modified_by")),
        "category_id": _to_str(old_document.get("doc_category_id")),
        "effective_status_id": _to_str(old_document.get("decree_status_id")),
        "type_id": _to_str(old_document.get("doc_type_id")),
        "issuing_level_id": _to_str(old_document.get("issuing_level_id")),
        "storage_id": _to_str(old_document.get("storage_id")),
        "agency_ids": _normalize_array(old_document.get("agency_ids")),
        "industry_sector_ids": _normalize_array(old_document.get("industry_sector_ids")),
        "keyword_ids": _normalize_array(old_document.get("keyword_ids")),
        "position_ids": _normalize_array(old_document.get("position_ids")),
        "signer_ids": _normalize_array(old_document.get("signer_ids")),
        "tree_ids": _normalize_array(old_document.get("tree_ids")),
        "status_in_system": "IN"
    }

def mapping_law_documents_upload(old_document):
    doc_short_description = _to_str(old_document.get("doc_short_description")).strip()
    if not doc_short_description:
        doc_short_description = _to_str(old_document.get("short_description")).strip()
    if not doc_short_description:
        doc_short_description = _to_str(old_document.get("doc_title"))
    return {
        "doc_id": _to_str(old_document.get("doc_id")),
        "doc_code": _to_str(old_document.get("doc_code")),
        "doc_title": _to_str(old_document.get("doc_title")),
        "doc_short_description": doc_short_description,
        "doc_content": _to_str(old_document.get("doc_content")),
        "doc_issue_date": _normalize_business_datetime(old_document.get("issue_date")),
        "doc_effective_date": _normalize_business_datetime(old_document.get("effective_date")),
        "doc_expiry_date": _normalize_business_datetime(old_document.get("end_effective_date")),
        "data_source": _to_str(old_document.get("data_source")),
        "created_at": _normalize_datetime(old_document.get("created_date")),
        "created_by": _to_str(old_document.get("created_by")),
        "last_modified_at": _normalize_datetime(old_document.get("last_modified")),
        "last_modified_by": _to_str(old_document.get("last_modified_by")),
        "category_id": _to_str(old_document.get("document_category_code")),
        "effective_status_id": _to_str(old_document.get("decree_status_code")),
        "type_id": _to_str(old_document.get("doc_type_id")),
        "issuing_level_id": _to_str(old_document.get("issued_level_code")),
        "storage_id": _to_str(old_document.get("storage_code")),
        "agency_ids": _normalize_array(old_document.get("agency_codes")),
        "industry_sector_ids": _normalize_array(old_document.get("industry_sector_codes")),
        "keyword_ids": _normalize_array(old_document.get("keyword_codes")),
        "position_ids": _normalize_array(old_document.get("position_codes")),
        "signer_ids": _normalize_array(old_document.get("signer_codes")),
        "tree_ids": _normalize_array(old_document.get("tree_codes")),
        "status_in_system": "OUT"
    }

def _streaming_parallel_migrate_documents(sources, target_collection, batch_size=5000, max_workers=None):
    if max_workers is None:
        max_workers = min(16, (os.cpu_count() or 4))
    def process_batch(batch, mapping_function):
        operations = []
        skipped_missing_key = 0
        skipped_invalid_payload = 0
        errors = []
        for old_document in batch:
            try:
                new_document = mapping_function(old_document)
                doc_id = _to_str(new_document.get("doc_id")).strip()
                if not doc_id:
                    skipped_missing_key += 1
                    continue
                if not _is_valid_law_document_payload(new_document):
                    skipped_invalid_payload += 1
                    continue
                operations.append(UpdateOne({"doc_id": doc_id}, {"$set": new_document}, upsert=True))
            except Exception as e:
                errors.append(str(e))
        if not operations:
            return {
                "processed": 0,
                "upserted": 0,
                "modified": 0,
                "matched": 0,
                "skipped_missing_key": skipped_missing_key,
                "skipped_invalid_payload": skipped_invalid_payload,
                "errors": errors
            }
        try:
            result = target_collection.bulk_write(operations, ordered=False)
            return {
                "processed": len(operations),
                "upserted": result.upserted_count,
                "modified": result.modified_count,
                "matched": result.matched_count,
                "skipped_missing_key": skipped_missing_key,
                "skipped_invalid_payload": skipped_invalid_payload,
                "errors": errors
            }
        except Exception as e:
            errors.append(str(e))
            return {
                "processed": len(operations),
                "upserted": 0,
                "modified": 0,
                "matched": 0,
                "skipped_missing_key": skipped_missing_key,
                "skipped_invalid_payload": skipped_invalid_payload,
                "errors": errors
            }
    overall = {
        "upload_total": 0,
        "main_total": 0,
        "processed_upload": 0,
        "processed_main": 0,
        "upserted_upload": 0,
        "upserted_main": 0,
        "modified_upload": 0,
        "modified_main": 0,
        "matched_upload": 0,
        "matched_main": 0,
        "skipped_missing_key_upload": 0,
        "skipped_missing_key_main": 0,
        "skipped_invalid_payload_upload": 0,
        "skipped_invalid_payload_main": 0,
        "errors": []
    }
    for source_name, source_collection, mapping_function in sources:
        total_docs = source_collection.count_documents({})
        if source_name == "upload":
            overall["upload_total"] = total_docs
        else:
            overall["main_total"] = total_docs
        futures = []
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            batch = []
            cursor = source_collection.find({}, no_cursor_timeout=True).batch_size(batch_size)
            try:
                for doc in cursor:
                    batch.append(doc)
                    if len(batch) >= batch_size:
                        futures.append(executor.submit(process_batch, batch, mapping_function))
                        batch = []
                if batch:
                    futures.append(executor.submit(process_batch, batch, mapping_function))
            finally:
                cursor.close()
            for future in as_completed(futures):
                result = future.result()
                overall["errors"].extend(result["errors"])
                if source_name == "upload":
                    overall["processed_upload"] += result["processed"]
                    overall["upserted_upload"] += result["upserted"]
                    overall["modified_upload"] += result["modified"]
                    overall["matched_upload"] += result["matched"]
                    overall["skipped_missing_key_upload"] += result["skipped_missing_key"]
                    overall["skipped_invalid_payload_upload"] += result["skipped_invalid_payload"]
                else:
                    overall["processed_main"] += result["processed"]
                    overall["upserted_main"] += result["upserted"]
                    overall["modified_main"] += result["modified"]
                    overall["matched_main"] += result["matched"]
                    overall["skipped_missing_key_main"] += result["skipped_missing_key"]
                    overall["skipped_invalid_payload_main"] += result["skipped_invalid_payload"]
    return overall

# Gộp biz_upload_documents và law_documents vào law_documents; bản chính thức overwrite bản upload nếu trùng doc_id.
target_collection.create_index([("doc_id", ASCENDING)], unique=True, name="uq_law_documents_doc_id")

result = _streaming_parallel_migrate_documents(
    sources=[
        ("upload", source_upload, mapping_law_documents_upload),
        ("main", source_main, mapping_law_documents_main)
    ],
    target_collection=target_collection,
    batch_size=5000,
    max_workers=min(16, (os.cpu_count() or 4))
)

print({
    "collection": TARGET_COLLECTION,
    "upload_total": result["upload_total"],
    "main_total": result["main_total"],
    "processed_upload": result["processed_upload"],
    "processed_main": result["processed_main"],
    "upserted_upload": result["upserted_upload"],
    "upserted_main": result["upserted_main"],
    "modified_upload": result["modified_upload"],
    "modified_main": result["modified_main"],
    "matched_upload": result["matched_upload"],
    "matched_main": result["matched_main"],
    "skipped_missing_key_upload": result["skipped_missing_key_upload"],
    "skipped_missing_key_main": result["skipped_missing_key_main"],
    "skipped_invalid_payload_upload": result["skipped_invalid_payload_upload"],
    "skipped_invalid_payload_main": result["skipped_invalid_payload_main"],
    "errors": len(result["errors"]),
    "target_total": target_collection.count_documents({})
})

if result["errors"]:
    display(pd.DataFrame(result["errors"], columns=["error"]))

## 6. law_document_storage

In [ ]:
SOURCE_DB = "v03_standardize_281125_dev"
SOURCE_COLLECTION = "resource"
TARGET_COLLECTION = "law_document_storage"
source_collection = client[SOURCE_DB][SOURCE_COLLECTION]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]

def mapping_law_document_storage(old_document):
    return {
        "storage_id": _to_str(old_document.get("code")),
        "bucket": _to_str(old_document.get("bucket")),
        "name": _to_str(old_document.get("name")),
        "path": _to_str(old_document.get("path")),
        "created_at": _normalize_datetime(old_document.get("created_date")),
        "created_by": _to_str(old_document.get("created_by"))
    }

result = parallel_migrate(
    sources=[source_collection],
    mapping_function=mapping_law_document_storage,
    upsert_key="storage_id",
    target_collection=target_collection,
    batch_size=2000,
    max_workers=16
)

print({
    "source_db": SOURCE_DB,
    "source_collection": SOURCE_COLLECTION,
    "target_collection": TARGET_COLLECTION,
    **result
})

/home/ubuntu/miniconda3/envs/v03/lib/python3.10/site-packages/pymongo/synchronous/collection.py:1945: UserWarning: use an explicit session with no_cursor_timeout=True otherwise the cursor may still timeout after 30 minutes, for more info see https://mongodb.com/docs/v4.4/reference/method/cursor.noCursorTimeout/#session-idle-timeout-overrides-nocursortimeout
  return Cursor(self, *args, **kwargs)


{'source_db': 'v03_standardize_281125_dev', 'source_collection': 'resource', 'target_collection': 'law_document_storage', 'source_total': 364177, 'submitted': 364177, 'processed': 364177, 'upserted': 364177, 'errors': 0, 'target_total': 364177}


## 7. law_articles

In [5]:
SOURCE_DB = "v03_core_281125_dev"
SOURCE_MAIN = "law_articles"
SOURCE_DRAFT = "law_articles_draft"
TARGET_COLLECTION = "law_articles"
source_main = client[SOURCE_DB][SOURCE_MAIN]
source_draft = client[SOURCE_DB][SOURCE_DRAFT]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]

def mapping_law_articles(old):
    return {
        "article_id": _to_str(old.get("article_id")),
        "doc_id": _to_str(old.get("doc_id")),
        "article_title": _to_str(old.get("article_title")),
        "article_content": _to_str(old.get("article_content")),
        "article_index": _to_int(old.get("article_index")),
        "start_article_index": _to_int(old.get("start_article_index")),
        "article_effective_date": _normalize_datetime(old.get("article_effective_date")),
        "article_expiry_date": _normalize_datetime(old.get("article_expiry_date")),
        "effective_status_id": _to_str(old.get("decree_status_id")),
        "part": _to_str(old.get("part")),
        "chapter": _to_str(old.get("chapter")),
        "section": _to_str(old.get("section")),
        "sub_section": _to_str(old.get("sub_section")),
        "created_at": _normalize_datetime(old.get("created_date")),
        "created_by": _to_str(old.get("created_by")),
        "last_modified_at": _normalize_datetime(old.get("last_modified")),
        "last_modified_by": _to_str(old.get("last_modified_by"))
    }

result = parallel_migrate(
    sources=[source_draft, source_main],
    mapping_function=mapping_law_articles,
    upsert_key="article_id",
    target_collection=target_collection,
    batch_size=2000,
    max_workers=min(16, (os.cpu_count() or 4) * 2)
)

print({
    "collection": TARGET_COLLECTION,
    **result
})

{'collection': 'law_articles', 'source_total': 161818, 'processed': 161818, 'errors': 0, 'target_total': 160671}


## 8. law_clauds

In [ ]:
TARGET_COLLECTION = "law_clauds"
source_collection = client[SOURCE_CORE_DB][TARGET_COLLECTION]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]

def mapping_law_clauds(old_document):
    return {
        "claud_id": _to_str(old_document.get("claud_id")),
        "article_id": _to_str(old_document.get("article_id")),
        "claud_content": _to_str(old_document.get("claud_content")),
        "claud_summary_content": _to_str(old_document.get("claud_summary_content")),
        "claud_order_index": _to_int(old_document.get("claud_order_index")),
        "created_at": _normalize_datetime(old_document.get("created_date")),
        "created_by": _to_str(old_document.get("created_by")),
        "last_modified_at": _normalize_datetime(old_document.get("last_modified")),
        "last_modified_by": _to_str(old_document.get("last_modified_by"))
    }

result = parallel_migrate(
    sources=[source_collection],
    mapping_function=mapping_law_clauds,
    upsert_key="claud_id",
    target_collection=target_collection,
    batch_size=1000,
    max_workers=min(8, os.cpu_count() or 4)
)

print({
    "collection": TARGET_COLLECTION,
    **result
})

## 9. law_references

In [4]:
SOURCE_DB = "v03_core_281125_dev"
SOURCE_MAIN = "law_references"
SOURCE_DRAFT = "law_reference_draft"
TARGET_COLLECTION = "law_references"
EFFECTIVE_STATUS_COLLECTION = "law_effective_status"

source_main = client[SOURCE_DB][SOURCE_MAIN]
source_draft = client[SOURCE_DB][SOURCE_DRAFT]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]
effective_status_collection = client[TARGET_CORE_DB][EFFECTIVE_STATUS_COLLECTION]


def mapping_law_references(old_document):
    reference_status_name = _to_str(old_document.get("reference_status")).strip()
    effective_status_id = effective_status_map.get(reference_status_name, unknown_effective_status_id)
    return {
        "reference_id": _to_str(old_document.get("reference_id")),
        "source_id": _to_str(old_document.get("source_id")),
        "target_id": _to_str(old_document.get("target_id")),
        "effective_status_id": effective_status_id,
        "reference_type": _to_str(old_document.get("reference_type")),
        "created_at": _normalize_datetime_reference(old_document.get("created_date")),
        "created_by": _to_str(old_document.get("created_by")),
        "last_modified_at": _normalize_datetime_reference(old_document.get("last_modified")),
        "last_modified_by": _to_str(old_document.get("last_modified_by"))
    }


# Gộp draft và main vào law_references; bản main overwrite draft nếu trùng reference_id.
target_collection.create_index([("reference_id", ASCENDING)], unique=True, name="uq_law_references_reference_id")

result = streaming_parallel_migrate(
    sources=[("draft", source_draft), ("main", source_main)],
    mapping_function=mapping_law_references,
    upsert_key="reference_id",
    target_collection=target_collection,
    batch_size=5000,
    max_workers=min(16, (os.cpu_count() or 4))
)

print({
    "collection": TARGET_COLLECTION,
    "draft_total": result["draft_total"],
    "main_total": result["main_total"],
    "processed_draft": result["processed_draft"],
    "processed_main": result["processed_main"],
    "upserted_draft": result["upserted_draft"],
    "upserted_main": result["upserted_main"],
    "modified_draft": result["modified_draft"],
    "modified_main": result["modified_main"],
    "matched_draft": result["matched_draft"],
    "matched_main": result["matched_main"],
    "skipped_missing_key_draft": result["skipped_missing_key_draft"],
    "skipped_missing_key_main": result["skipped_missing_key_main"],
    "errors": len(result["errors"]),
    "target_total": target_collection.count_documents({})
})

if result["errors"]:
    display(pd.DataFrame(result["errors"], columns=["error"]))

/home/ubuntu/miniconda3/envs/v03/lib/python3.10/site-packages/pymongo/synchronous/collection.py:1945: UserWarning: use an explicit session with no_cursor_timeout=True otherwise the cursor may still timeout after 30 minutes, for more info see https://mongodb.com/docs/v4.4/reference/method/cursor.noCursorTimeout/#session-idle-timeout-overrides-nocursortimeout
  return Cursor(self, *args, **kwargs)


{'collection': 'law_references', 'draft_total': 155, 'main_total': 3673677, 'processed_draft': 155, 'processed_main': 3673677, 'upserted_draft': 155, 'upserted_main': 3673677, 'modified_draft': 0, 'modified_main': 0, 'matched_draft': 0, 'matched_main': 0, 'skipped_missing_key_draft': 0, 'skipped_missing_key_main': 0, 'errors': 0, 'target_total': 3673832}


## 10. law_references_article

In [6]:
SOURCE_DB = "v03_core_281125_dev"
SOURCE_MAIN = "law_references_article"
SOURCE_DRAFT = "law_references_article_draft"
TARGET_COLLECTION = "law_references_article"
source_main = client[SOURCE_DB][SOURCE_MAIN]
source_draft = client[SOURCE_DB][SOURCE_DRAFT]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]

def _get_draft_relationships(old_document):
    relationships = old_document.get("relationships")
    if isinstance(relationships, dict):
        return relationships
    return {}

def _get_first_detail(old_document):
    relationships = _get_draft_relationships(old_document)
    details = relationships.get("details")
    if isinstance(details, list) and details:
        first_detail = details[0]
        if isinstance(first_detail, dict):
            return first_detail
    return {}

def _is_valid_draft_reference_article(old_document):
    relationships = _get_draft_relationships(old_document)
    if relationships.get("error"):
        return False
    relationship_id = _to_str(old_document.get("relationship_id")).strip()
    if not relationship_id:
        return False
    return True

def mapping_law_references_article_main(old_document):
    return {
        "relationship_id": _to_str(old_document.get("relationship_id")),
        "source_doc_id": _to_str(old_document.get("source_doc_id")),
        "source_article_id": _to_str(old_document.get("source_article_id")),
        "source_clause": _to_str(old_document.get("source_clause")),
        "source_point": _to_str(old_document.get("source_point")),
        "target_doc_id": _to_str(old_document.get("target_doc_id")),
        "target_article_id": _to_str(old_document.get("target_article_id")),
        "target_article": _to_str(old_document.get("target_article")),
        "target_clause": _to_str(old_document.get("target_clause")),
        "target_point": _to_str(old_document.get("target_point")),
        "relationship_type": _to_str(old_document.get("relationship_type")),
        "created_at": _normalize_datetime(old_document.get("created_date")),
        "created_by": _to_str(old_document.get("created_by")),
        "last_modified_at": _normalize_datetime(old_document.get("last_modified")),
        "last_modified_by": _to_str(old_document.get("last_modified_by"))
    }

def mapping_law_references_article_draft(old_document):
    first_detail = _get_first_detail(old_document)
    relationships = _get_draft_relationships(old_document)
    relationship_type = _to_str(old_document.get("relationship_type"))
    if not relationship_type:
        relationship_type = _to_str(relationships.get("type_rel"))
    target_article = _to_str(old_document.get("target_article"))
    if not target_article:
        target_article = _to_str(first_detail.get("detail_article"))
    target_clause = _to_str(old_document.get("target_clause"))
    if not target_clause:
        target_clause = _to_str(first_detail.get("detail_clause"))
    target_point = _to_str(old_document.get("target_point"))
    if not target_point:
        target_point = _to_str(first_detail.get("detail_point"))
    return {
        "relationship_id": _to_str(old_document.get("relationship_id")),
        "source_doc_id": _to_str(old_document.get("source_doc_id")),
        "source_article_id": _to_str(old_document.get("source_article_id")) or _to_str(old_document.get("article_id")),
        "source_clause": _to_str(old_document.get("source_clause")),
        "source_point": _to_str(old_document.get("source_point")),
        "target_doc_id": _to_str(old_document.get("target_doc_id")),
        "target_article_id": _to_str(old_document.get("target_article_id")),
        "target_article": target_article,
        "target_clause": target_clause,
        "target_point": target_point,
        "relationship_type": relationship_type,
        "created_at": _normalize_datetime(old_document.get("created_date")),
        "created_by": _to_str(old_document.get("created_by")),
        "last_modified_at": _normalize_datetime(old_document.get("last_modified") or old_document.get("last_modified_date")),
        "last_modified_by": _to_str(old_document.get("last_modified_by"))
    }

def parallel_migrate_filtered(sources, mapping_function, upsert_key, target_collection, filter_function=None, batch_size=2000, max_workers=8):
    def wrapped_mapping(old_document):
        if filter_function is not None and not filter_function(old_document):
            return {}
        return mapping_function(old_document)
    return parallel_migrate(
        sources=sources,
        mapping_function=wrapped_mapping,
        upsert_key=upsert_key,
        target_collection=target_collection,
        batch_size=batch_size,
        max_workers=max_workers
    )

# Gộp draft và main vào law_references_article; bản main overwrite draft nếu trùng relationship_id.
target_collection.create_index([("relationship_id", ASCENDING)], unique=True, name="uq_law_references_article_relationship_id")

draft_result = parallel_migrate_filtered(
    sources=[source_draft],
    mapping_function=mapping_law_references_article_draft,
    upsert_key="relationship_id",
    target_collection=target_collection,
    filter_function=_is_valid_draft_reference_article,
    batch_size=2000,
    max_workers=8
)

main_result = parallel_migrate(
    sources=[source_main],
    mapping_function=mapping_law_references_article_main,
    upsert_key="relationship_id",
    target_collection=target_collection,
    batch_size=2000,
    max_workers=8
)

print({
    "collection": TARGET_COLLECTION,
    "draft_total": draft_result["source_total"],
    "main_total": main_result["source_total"],
    "processed_draft": draft_result["processed"],
    "processed_main": main_result["processed"],
    "errors_draft": draft_result["errors"],
    "errors_main": main_result["errors"],
    "target_total": target_collection.count_documents({})
})

{'collection': 'law_references_article', 'draft_total': 1413, 'main_total': 136533, 'processed_draft': 23, 'processed_main': 136533, 'errors_draft': 0, 'errors_main': 0, 'target_total': 136556}


## 11. law_agencies

In [6]:
SOURCE_DB = "v03_core_281125_dev"
SOURCE_COLLECTION = "law_agencies"

TARGET_COLLECTION = "law_agencies"

source_collection = client[SOURCE_DB][SOURCE_COLLECTION]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]
    
# Migrate danh mục cơ quan; loại bỏ các field deprecated như description

def mapping_law_agencies(old):
    status = _to_str(old.get("status")).upper()
    if not status:
        status = "ACTIVE"
    return {
        "agency_id": _to_str(old.get("agency_id")),
        "agency_name": _to_str(old.get("agency_name")),
        "status": status,
        "created_at": _normalize_datetime(old.get("created_date")),
        "created_by": _to_str(old.get("created_by")),
        "last_modified_at": _normalize_datetime(old.get("last_modified")),
        "last_modified_by": _to_str(old.get("last_modified_by"))
    }


result = parallel_migrate(
    sources=[source_collection],
    mapping_function=mapping_law_agencies,
    upsert_key="agency_id",
    target_collection=target_collection,
    batch_size=1000,   
    max_workers=4
)


print({
    "collection": TARGET_COLLECTION,
    **result
})

{'collection': 'law_agencies', 'source_total': 1168, 'processed': 1168, 'errors': 0, 'target_total': 1168}


## 12. law_article_class

In [ ]:
SOURCE_DB = "v03_core_281125_dev"
SOURCE_COLLECTION = "law_articles_class"
TARGET_COLLECTION = "law_article_class"
source_collection = client[SOURCE_DB][SOURCE_COLLECTION]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]

def _normalize_string_array(values):
    if not isinstance(values, list):
        return [""]
    cleaned = []
    for value in values:
        value_str = _to_str(value).strip()
        if value_str:
            cleaned.append(value_str)
    return cleaned if cleaned else [""]

def mapping_law_article_class(old):
    now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return {
        "article_id": _to_str(old.get("article_id")),
        "doc_id": _to_str(old.get("doc_id")),
        "article_title": _to_str(old.get("article_title")),
        "article_content": _to_str(old.get("article_content")),
        "class": _normalize_string_array(old.get("class")),
        "version": _normalize_string_array(old.get("version")),
        "created_at": now_str,
        "created_by": _to_str(old.get("created_by")),
        "last_modified_at": now_str,
        "last_modified_by": _to_str(old.get("created_by"))
    }

# Migrate kết quả phân loại AI của điều luật từ law_articles_class.
# Không migrate các field deprecated như check_social_relations.

result = parallel_migrate(
    sources=[source_collection],
    mapping_function=mapping_law_article_class,
    upsert_key="article_id",
    target_collection=target_collection,
    batch_size=2000,
    max_workers=8
)

print({
    "collection": TARGET_COLLECTION,
    **result
})

## 13. + 14. law_authority & law_authority_mapping

In [5]:
SOURCE_DB = "v03_core_281125_dev"
SOURCE_AUTHORITY_COLLECTION = "law_authority"
SOURCE_AUTHORITY_MAPPING_COLLECTION = "law_authority_mapping"
TARGET_AUTHORITY_COLLECTION = "law_authority"
TARGET_AUTHORITY_MAPPING_COLLECTION = "law_authority_mapping"

source_authority_collection = client[SOURCE_DB][SOURCE_AUTHORITY_COLLECTION]
source_authority_mapping_collection = client[SOURCE_DB][SOURCE_AUTHORITY_MAPPING_COLLECTION]
target_authority_collection = client[TARGET_CORE_DB][TARGET_AUTHORITY_COLLECTION]
target_authority_mapping_collection = client[TARGET_CORE_DB][TARGET_AUTHORITY_MAPPING_COLLECTION]

def _normalize_optional_datetime(value):
    return _format_datetime(value)

def mapping_law_authority(old_document):
    effective_status_name = _to_str(old_document.get("doc_effective_status")).strip()
    effective_status_id = effective_status_map.get(effective_status_name, effective_status_map.get("Không xác định", ""))
    return {
        "authority_id": _to_str(old_document.get("authority_id")),
        "authority_content": _to_str(old_document.get("authority_content")),
        "doc_effective_date": _normalize_optional_datetime(old_document.get("doc_effective_date")),
        "doc_expiry_date": _normalize_optional_datetime(old_document.get("doc_expire_date")),
        "effective_status_id": effective_status_id,
        "status": _to_str(old_document.get("status")).upper() or "ACTIVE",
        "created_at": _normalize_datetime(old_document.get("created_date")),
        "created_by": _to_str(old_document.get("created_by")),
        "last_modified_at": _normalize_datetime(old_document.get("last_modified")),
        "last_modified_by": _to_str(old_document.get("last_modified_by"))
    }

def mapping_law_authority_mapping(old_document):
    return {
        "authority_id": _to_str(old_document.get("authority_id")),
        "doc_id": _to_str(old_document.get("doc_id")),
        "article_id": _to_str(old_document.get("article_id")),
        "agency_id": _to_str(old_document.get("agency_id")),
        "created_at": _normalize_datetime(old_document.get("created_date")),
        "created_by": _to_str(old_document.get("created_by")),
        "last_modified_at": _normalize_datetime(old_document.get("last_modified")),
        "last_modified_by": _to_str(old_document.get("last_modified_by"))
    }

# Migrate law_authority từ collection chính; bỏ doc_id khỏi bản ghi authority chuẩn.
# Bỏ qua law_authority_draft vì dữ liệu draft lỗi, không có mapping tương ứng nên không migrate.
authority_result = parallel_migrate(
    sources=[source_authority_collection],
    mapping_function=mapping_law_authority,
    upsert_key="authority_id",
    target_collection=target_authority_collection,
    batch_size=1000,
    max_workers=8
)

# Migrate law_authority_mapping; collection này chịu trách nhiệm ánh xạ authority tới văn bản/điều luật/cơ quan ban hành.
# Theo rule hiện tại chỉ upsert theo authority_id, không dùng composite key.
authority_mapping_result = parallel_migrate(
    sources=[source_authority_mapping_collection],
    mapping_function=mapping_law_authority_mapping,
    upsert_key="authority_id",
    target_collection=target_authority_mapping_collection,
    batch_size=1000,
    max_workers=8
)

print({
    "law_authority": authority_result,
    "law_authority_mapping": authority_mapping_result
})

{'law_authority': {'source_total': 1054, 'processed': 1054, 'errors': 0, 'target_total': 1054}, 'law_authority_mapping': {'source_total': 1059, 'processed': 1059, 'errors': 0, 'target_total': 1050}}


## 15. law_authority_prompts 

In [4]:
SOURCE_DB = "v03_core_281125_dev"
SOURCE_COLLECTION = "law_authority_prompts"
TARGET_COLLECTION = "law_authority_prompts"
source_collection = client[SOURCE_DB][SOURCE_COLLECTION]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]

def mapping_law_authority_prompts(old_document):
    return {
        "agency": _to_str(old_document.get("agency")),
        "doc_id": _to_str(old_document.get("doc_id")),
        "authority_check_prompt": _to_str(old_document.get("authority_check_prompt"))
    }

# Upsert theo (agency, doc_id); nếu trùng key này thì bản ghi mới sẽ overwrite bản ghi cũ.
result = parallel_migrate_composite_key(
    sources=[source_collection],
    mapping_function=mapping_law_authority_prompts,
    upsert_filter_function=lambda doc: {"agency": doc["agency"], "doc_id": doc["doc_id"]} if doc["agency"] else None,
    target_collection=target_collection,
    batch_size=500,
    max_workers=4
)

print({
    "collection": TARGET_COLLECTION,
    **result
})

{'collection': 'law_authority_prompts', 'source_total': 172, 'processed': 172, 'errors': 0, 'target_total': 172}


/home/ubuntu/miniconda3/envs/v03/lib/python3.10/site-packages/pymongo/synchronous/collection.py:1945: UserWarning: use an explicit session with no_cursor_timeout=True otherwise the cursor may still timeout after 30 minutes, for more info see https://mongodb.com/docs/v4.4/reference/method/cursor.noCursorTimeout/#session-idle-timeout-overrides-nocursortimeout
  return Cursor(self, *args, **kwargs)


## 16. law_core_models

In [4]:
SOURCE_DB = "v03_core_281125_dev"
SOURCE_COLLECTION = "law_core_models"
TARGET_COLLECTION = "law_core_models"
source_collection = client[SOURCE_DB][SOURCE_COLLECTION]
target_collection = client[TARGET_CORE_DB][TARGET_COLLECTION]

def mapping_law_core_models(old_document):
    return {
        "model_id": _to_str(old_document.get("model_id")),
        "model_name": _to_str(old_document.get("model_name")),
        "model_type": _to_str(old_document.get("model_type")),
        "organization": _to_str(old_document.get("organization")),
        "framework": _to_str(old_document.get("framework")),
        "status": _to_str(old_document.get("status")).upper() or "ACTIVE",
        "availability": _to_str(old_document.get("availability")).upper(),
        "is_trained": _normalize_bool(old_document.get("is_trained")),
        "description": _to_str(old_document.get("description")),
        "created_at": _normalize_datetime(old_document.get("createdAt")),
        "last_modified_at": _normalize_datetime(old_document.get("updatedAt")),
        "config": _normalize_json_object(old_document.get("config"))
    }

# Migrate model registry từ law_core_models, chuẩn hóa createdAt/updatedAt và chuẩn hóa status/availability về IN HOA.
result = parallel_migrate(
    sources=[source_collection],
    mapping_function=mapping_law_core_models,
    upsert_key="model_id",
    target_collection=target_collection,
    batch_size=100,
    max_workers=4
)

print({
    "collection": TARGET_COLLECTION,
    **result
})

{'collection': 'law_core_models', 'source_total': 25, 'processed': 0, 'errors': 0, 'target_total': 25}


/home/ubuntu/miniconda3/envs/v03/lib/python3.10/site-packages/pymongo/synchronous/collection.py:1945: UserWarning: use an explicit session with no_cursor_timeout=True otherwise the cursor may still timeout after 30 minutes, for more info see https://mongodb.com/docs/v4.4/reference/method/cursor.noCursorTimeout/#session-idle-timeout-overrides-nocursortimeout
  return Cursor(self, *args, **kwargs)
